In [5]:
from pykrx import stock
import pandas as pd
from datetime import datetime

### **ETF**

#### 1. Ticker-name keyword-based mapping (Rule-based)

In [6]:
# 1. 조회할 날짜 지정 (오늘)
today = datetime.today().strftime("%Y%m%d")

# 2. 위험 등급별 키워드 딕셔너리 정의 (우선순위를 위해 높은 등급부터 배치)
# 리스트 안의 키워드 중 하나라도 포함되면 해당 등급으로 판별합니다.
risk_rules = {
    5: ["레버리지", "인버스", "2X", "바이오", "2차전지"],
    4: ["200", "S&P500", "나스닥100"],
    3: ["고배당", "배당성장", "커버드콜"],
    2: ["국고채", "단기채", "통안채", "달러단기"]
}

# 3. ETF 티커 리스트 가져오기
tickers = stock.get_etf_ticker_list(today)
print(f"총 {len(tickers)}개의 ETF 데이터를 불러오고 분석합니다. 잠시만 기다려주세요...")

# 4. 종목별로 이름을 가져와서 등급 매핑하기
etf_list = []

for ticker in tickers:
    name = stock.get_etf_ticker_name(ticker)
    risk_level = "미분류"  # 설정한 키워드에 해당하지 않는 경우의 기본값
    
    # 등급을 5 -> 4 -> 3 -> 2 순으로 내림차순 검사
    for level in sorted(risk_rules.keys(), reverse=True):
        keywords = risk_rules[level]
        
        # 종목명에 해당 등급의 키워드가 하나라도 포함되어 있는지 확인
        if any(keyword in name for keyword in keywords):
            risk_level = level
            break  # 등급이 결정되면 하위 등급은 검사하지 않고 루프 탈출
            
    # 결과 저장
    etf_list.append({
        "Ticker": ticker,
        "Name": name,
        "Risk_Level": risk_level
    })

# 5. 결과를 Pandas DataFrame으로 변환 (보기 좋게 만들기)
df_etf = pd.DataFrame(etf_list)

# 결과 출력 (상위 10개)
df_etf.head(10)

총 1099개의 ETF 데이터를 불러오고 분석합니다. 잠시만 기다려주세요...


,Ticker,Name,Risk_Level
0,495710,BNK 26-06 특수채(AAA이상)액티브,미분류
1,466810,BNK 2차전지양극재,5
2,457930,BNK 미래전략기술액티브,미분류
3,487750,BNK 온디바이스AI,미분류
4,445690,BNK 주주가치액티브,미분류
5,0120J0,BNK 카카오그룹포커스,미분류
6,0112X0,마이티 200TR,4
7,465780,마이티 26-09 특수채(AAA)액티브,미분류
8,442260,마이티 다이나믹퀀트액티브,미분류
9,0001P0,마이티 바이오시밀러&CDMO액티브,5


In [10]:
# 1. 특정 위험도 ETF만 모아보기
df_etf[df_etf['Risk_Level'] == 2]

,Ticker,Name,Risk_Level
36,385560,RISE KIS국고채30년Enhanced,2
47,481430,RISE 국고채10년액티브,2
49,114100,RISE 국고채3년,2
70,385550,RISE 단기채권알파액티브,2
71,196230,RISE 단기통안채,2
159,448490,HANARO 32-10 국고채액티브,2
274,272580,TIGER 단기채권액티브,2
275,157450,TIGER 단기통안채,2
304,329750,TIGER 미국달러단기채권액티브,2
412,451530,TIGER 국고채30년스트립액티브,2


In [ ]:
# 2. 등급별로 ETF가 몇 개씩 분류되었는지 통계 확인하기
df_etf['Risk_Level'].value_counts()

Risk_Level
미분류    742
4      130
5      119
3       72
2       36
Name: count, dtype: int64

In [ ]:
# 3. '미분류'로 빠진 ETF들 확인하기 (키워드 규칙 업데이트 용도)
df_unclassified = df_etf[df_etf['Risk_Level'] == '미분류']
df_unclassified.head(20)

,Ticker,Name,Risk_Level
0,495710,BNK 26-06 특수채(AAA이상)액티브,미분류
2,457930,BNK 미래전략기술액티브,미분류
3,487750,BNK 온디바이스AI,미분류
4,445690,BNK 주주가치액티브,미분류
5,0120J0,BNK 카카오그룹포커스,미분류
7,465780,마이티 26-09 특수채(AAA)액티브,미분류
8,442260,마이티 다이나믹퀀트액티브,미분류
10,159800,마이티 코스피100,미분류
12,0005G0,ITF K-AI반도체코어테크,미분류
13,0178H0,ITF 미국AI TOP10국채혼합50,미분류


#### 2. volatility based

In [12]:
from pykrx import stock
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time

# 1. 날짜 설정 (오늘부터 정확히 1년 전)
today = datetime.today()
one_year_ago = today - relativedelta(years=1)

end_date = today.strftime("%Y%m%d")
start_date = one_year_ago.strftime("%Y%m%d")

# 2. 전체 ETF 티커 리스트 가져오기
all_tickers = stock.get_etf_ticker_list(end_date)
print(f"전체 ETF 개수: {len(all_tickers)}개")

# 테스트를 위해 우선 10개만 실행해 봅니다. (전체 실행 시 [:10]을 지워주세요!)
test_tickers = all_tickers 
print(f"이 중 {len(test_tickers)}개의 변동성을 계산합니다...\n")

etf_volatility_list = []

# 3. 데이터 수집 및 변동성 계산 루프
for i, ticker in enumerate(test_tickers):
    name = stock.get_etf_ticker_name(ticker)
    
    # 1년치 OHLCV 데이터 수집
    df = stock.get_etf_ohlcv_by_date(start_date, end_date, ticker)
    
    # 상장된 지 1년이 안 되어 데이터가 너무 적은(예: 120일 미만) ETF는 제외
    if len(df) < 120:
        print(f"[{i+1}/{len(test_tickers)}] {name} - 상장 기간 부족으로 패스")
        continue
        
    # 일일 수익률 계산 (pct_change)
    daily_returns = df['종가'].pct_change()
    
    # 연환산 변동성 계산 (일일 수익률 표준편차 * 루트 252)
    # np.sqrt(252)는 1년 영업일(약 252일)을 의미하는 금융공학 기본 상수입니다.
    annual_volatility = daily_returns.std() * np.sqrt(252)
    
    # 퍼센트로 변환 (예: 0.15 -> 15.0)
    vol_percent = annual_volatility * 100
    
    # 4. 변동성 수치에 따른 등급 부여 (Rule 기반)
    if vol_percent < 5.0:
        risk_level = 1  # 초저위험
    elif vol_percent < 10.0:
        risk_level = 2  # 저위험
    elif vol_percent < 15.0:
        risk_level = 3  # 중위험
    elif vol_percent < 25.0:
        risk_level = 4  # 고위험
    else:
        risk_level = 5  # 초고위험
        
    etf_volatility_list.append({
        "Ticker": ticker,
        "Name": name,
        "Volatility(%)": round(vol_percent, 2),
        "Risk_Level": risk_level
    })
    
    print(f"[{i+1}/{len(test_tickers)}] {name} - 변동성: {vol_percent:.2f}% (등급: {risk_level})")
    
    # 서버 과부하 및 IP 차단 방지를 위해 0.5초 대기 (매우 중요!)
    time.sleep(0.5)

# 5. 결과를 데이터프레임으로 변환
df_volatility = pd.DataFrame(etf_volatility_list)

print("\n계산 완료!")

전체 ETF 개수: 1099개
이 중 1099개의 변동성을 계산합니다...

[1/1099] BNK 26-06 특수채(AAA이상)액티브 - 변동성: 0.88% (등급: 1)
[2/1099] BNK 2차전지양극재 - 변동성: 54.38% (등급: 5)
[3/1099] BNK 미래전략기술액티브 - 변동성: 35.93% (등급: 5)
[4/1099] BNK 온디바이스AI - 변동성: 48.53% (등급: 5)
[5/1099] BNK 주주가치액티브 - 변동성: 30.36% (등급: 5)
[6/1099] BNK 카카오그룹포커스 - 상장 기간 부족으로 패스
[7/1099] 마이티 200TR - 변동성: 49.09% (등급: 5)
[8/1099] 마이티 26-09 특수채(AAA)액티브 - 변동성: 2.77% (등급: 1)
[9/1099] 마이티 다이나믹퀀트액티브 - 변동성: 38.75% (등급: 5)
[10/1099] 마이티 바이오시밀러&CDMO액티브 - 변동성: 36.59% (등급: 5)
[11/1099] 마이티 코스피100 - 변동성: 38.30% (등급: 5)
[12/1099] ITF 200 - 변동성: 39.16% (등급: 5)
[13/1099] ITF K-AI반도체코어테크 - 변동성: 46.88% (등급: 5)
[14/1099] ITF 미국AI TOP10국채혼합50 - 상장 기간 부족으로 패스
[15/1099] ITF 중기종합채권(AA-이상)액티브 - 상장 기간 부족으로 패스
[16/1099] RISE 200TR - 변동성: 37.21% (등급: 5)
[17/1099] RISE 200고배당커버드콜ATM - 변동성: 19.10% (등급: 4)
[18/1099] RISE 200금융 - 변동성: 32.83% (등급: 5)
[19/1099] RISE 200선물레버리지 - 변동성: 74.97% (등급: 5)
[20/1099] RISE 200선물인버스2X - 변동성: 73.48% (등급: 5)
[21/1099] RISE 200선물인버스 - 변동성: 37.57% (등급: 5)

KeyboardInterrupt: 

### **Blue-chip Dividend Stocks**

In [13]:
from pykrx import stock
import pandas as pd
from datetime import datetime, timedelta

# 1. 가장 최근 거래일 찾기 (주말/공휴일 대응)
today = datetime.today()
for i in range(10):
    date_str = (today - timedelta(days=i)).strftime("%Y%m%d")
    df_cap = stock.get_market_cap_by_ticker(date_str, market="KOSPI")
    if not df_cap.empty:
        target_date = date_str
        break

print(f"기준일자: {target_date} (우량 배당주 검색 중...)")

# 2. 시가총액 및 펀더멘털(DIV, PER, PBR 등) 데이터 불러오기
df_cap = stock.get_market_cap_by_ticker(target_date, market="KOSPI")
df_fund = stock.get_market_fundamental_by_ticker(target_date, market="KOSPI")

# 3. 두 데이터프레임 병합 (티커 기준)
df_merged = pd.merge(df_cap, df_fund, left_index=True, right_index=True)

# 4. 시가총액 순으로 내림차순 정렬하여 상위 100위(우량주)만 자르기
df_top100 = df_merged.sort_values(by="시가총액", ascending=False).head(100)

# 5. 배당수익률(DIV)이 4% 이상인 종목 필터링
df_dividend = df_top100[df_top100['DIV'] >= 4.0].copy()

# 6. 보기 편하게 종목명 추가 및 컬럼 정리
df_dividend['종목명'] = [stock.get_market_ticker_name(ticker) for ticker in df_dividend.index]

# 결과 확인 (시가총액, 종가, DIV, PER 위주로 출력)
df_dividend = df_dividend[['종목명', '종가', '시가총액', 'DIV', 'PER', 'PBR']]
df_dividend.sort_values(by="DIV", ascending=False).reset_index(drop=True)

기준일자: 20260504 (우량 배당주 검색 중...)


,종목명,종가,시가총액,DIV,PER,PBR
0,기업은행,22100,17623111704900,4.74,6.94,0.48
1,DB손해보험,166800,10925458213200,4.56,5.64,0.93
2,기아,154000,60123601692000,4.42,7.95,0.98
3,LG유플러스,15990,6872955396450,4.13,13.10,0.77
4,삼성화재,473500,21140578465500,4.12,9.97,0.95
5,우리금융지주,33300,24444741456000,4.08,8.22,0.68
6,현대차2우B,249500,8723500535000,4.05,0.00,0.00


### **Mid-cap Tech Stocks**

In [17]:
from pykrx import stock
from datetime import datetime, timedelta

# 최근 영업일 찾기
today = datetime.today()
for i in range(10):
    target_date = (today - timedelta(days=i)).strftime("%Y%m%d")
    if not stock.get_market_cap_by_ticker(target_date, market="KOSPI").empty:
        break

print(f"=== 기술주 관련 지수 검색 (기준일: {target_date}) ===")
markets = ["KOSPI", "KOSDAQ", "KRX"]
keywords = ["IT", "반도체", "소프트웨어", "기술", "게임", "디지털", "컴퓨터"]

for market in markets:
    tickers = stock.get_index_ticker_list(target_date, market=market)
    for ticker in tickers:
        name = stock.get_index_ticker_name(ticker)
        # 키워드 중 하나라도 포함된 지수만 출력
        if any(keyword in name for keyword in keywords):
            print(f"[{market}] {name} (티커: {ticker})")

=== 기술주 관련 지수 검색 (기준일: 20260504) ===
[KOSPI] IT 서비스 (티커: 1046)
[KOSPI] 코스피 200 정보기술 (티커: 1155)
[KOSDAQ] IT 서비스 (티커: 2118)
[KOSDAQ] 코스닥 기술성장기업부 (티커: 2184)
[KOSDAQ] 코스닥 150 정보기술 (티커: 2216)
[KRX] KRX 반도체 (티커: 5044)
[KRX] KRX 정보기술 (티커: 5064)
[KRX] KRX 300 정보기술 (티커: 5351)


In [36]:
from pykrx import stock
import pandas as pd
from datetime import datetime, timedelta

# 📌 선택한 지수 티커 입력
TARGET_INDEX_TICKER = "5351"

print(f"선택한 지수 티커: {TARGET_INDEX_TICKER}")

today = datetime.today()
it_tickers = []
target_date = ""

# 1. 과거 10일을 뒤지면서 편입 종목 가져오기
for i in range(10):
    test_date = (today - timedelta(days=i)).strftime("%Y%m%d")
    try:
        # 🚨 수정된 핵심 포인트: (티커, 날짜) 순서로 넣어야 합니다!
        tickers = stock.get_index_portfolio_deposit_file(TARGET_INDEX_TICKER, test_date)
        
        if len(tickers) > 0:
            it_tickers = tickers
            target_date = test_date
            break
    except:
        pass

if len(it_tickers) == 0:
    raise ValueError("❌ 최근 10일간의 데이터를 뒤져도 편입 종목이 나오지 않습니다. 티커 번호를 다시 확인해주세요.")

print(f"✅ 데이터 기준일: {target_date}")
print(f"✅ 편입 종목 수: {len(it_tickers)}개\n")

# 2. 시장은 "ALL"로 고정 (코스피, 코스닥 종목을 한 번에 다 불러옵니다)
market_type = "ALL" 

# 3. 전체 시장 시총 및 펀더멘털 가져오기
df_cap = stock.get_market_cap_by_ticker(target_date, market=market_type)
df_fund = stock.get_market_fundamental_by_ticker(target_date, market=market_type)

# 4. 데이터프레임 병합
df_tech = pd.DataFrame(index=it_tickers)
df_tech = df_tech.join(df_cap).join(df_fund)

df_tech = df_tech.dropna(subset=['시가총액'])

# 5. 중소형주 필터링 (시총 상위 대장주 제외)
df_tech = df_tech.sort_values(by="시가총액", ascending=False)

# 💡 중요: 1046(IT 서비스)처럼 지수 전체 구성 종목이 30개 남짓으로 적을 경우, 
# iloc[15:]로 15개나 잘라버리면 남는 종목이 너무 적거나 없을 수 있습니다.
# 그래서 대장주 컷을 15개에서 5개로 줄였습니다! (결과를 보고 조절하세요)
df_mid_small_tech = df_tech.iloc[5:].copy() 

# 6. 밸류에이션 필터링 (PER 0 초과, 20 이하인 흑자 기업)
condition = (df_mid_small_tech['PER'] > 0) & (df_mid_small_tech['PER'] <= 20)
df_final_tech = df_mid_small_tech[condition].copy()

# 7. 보기 좋게 정리
df_final_tech['종목명'] = [stock.get_market_ticker_name(t) for t in df_final_tech.index]
df_final_tech = df_final_tech[['종목명', '종가', '시가총액', 'PER', 'PBR', 'DIV']]

# 결과 출력
df_final_tech.reset_index(drop=True).head(20)

선택한 지수 티커: 5351
✅ 데이터 기준일: 20260504
✅ 편입 종목 수: 42개



,종목명,종가,시가총액,PER,PBR,DIV
0,삼성에스디에스,166900,12914354820000,17.00,1.30,1.91
1,LG씨엔에스,65900,6384783973200,14.49,2.18,2.81
2,LX세미콘,63100,1026277330000,12.42,0.91,2.38
3,카페24,25600,620878182400,15.85,2.33,0.00
